<a href="https://colab.research.google.com/github/iffathsaleem/FYP_Project/blob/main/Home_Credit_Risk_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import roc_auc_score
from sklearn.feature_selection import mutual_info_classif
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
sns.set_theme(style="whitegrid")

In [ ]:
# Define Data Path
BASE_PATH = Path("/content/drive/MyDrive/FYP/Dataset/Home Credit Risk")

In [ ]:
# Load All Datasets from CSV
print("Loading datasets...")

application_train = pd.read_csv(BASE_PATH / "application_train.csv")
application_test = pd.read_csv(BASE_PATH / "application_test.csv")
bureau = pd.read_csv(BASE_PATH / "bureau.csv")
bureau_balance = pd.read_csv(BASE_PATH / "bureau_balance.csv")
previous_application = pd.read_csv(BASE_PATH / "previous_application.csv")
installments = pd.read_csv(BASE_PATH / "installments_payments.csv")
pos_cash = pd.read_csv(BASE_PATH / "POS_CASH_balance.csv")
credit_card = pd.read_csv(BASE_PATH / "credit_card_balance.csv")

# Added encoding='latin-1' to handle UnicodeDecodeError
column_description = pd.read_csv(BASE_PATH / "HomeCredit_columns_description.csv", encoding='latin-1')

print("All datasets loaded!")

In [ ]:
# Organize into Dictionary
datasets = {
    "application_train": application_train,
    "application_test": application_test,
    "bureau": bureau,
    "bureau_balance": bureau_balance,
    "previous_application": previous_application,
    "installments_payments": installments,
    "POS_CASH_balance": pos_cash,
    "credit_card_balance": credit_card
}

## Part 1: Dataset Understanding EDA

### 1.1 Dataset Size Overview

In [ ]:
print("\n" + "="*80)
print("PART 1: DATASET UNDERSTANDING EDA")
print("="*80)

dataset_summary = []
for name, df in datasets.items():
    dataset_summary.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Duplicate Rows": df.duplicated().sum(),
        "Missing Cells %": round(df.isna().sum().sum() / df.size * 100, 2)
    })

dataset_summary_df = pd.DataFrame(dataset_summary)
print("\nTable 1: Dataset Size Summary")
display(dataset_summary_df)

### 1.2 Inspect First Records

In [ ]:
print("\n1.2 First Records Preview")
for name, df in datasets.items():
    print("\n" + "="*70)
    print(f"Dataset: {name}")
    print("="*70)
    display(df.head(3))

### 1.3 Data Types Analysis

In [ ]:
print("\n1.3 Data Types Summary")
for name, df in datasets.items():
    print(f"\n{name}:")
    print(df.dtypes.value_counts())

print("\nApplication Train - Detailed Info:")
print(f"Total columns: {len(application_train.columns)}")
print(f"Total rows: {application_train.shape[0]:,}")

numeric_cols = application_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = application_train.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Numeric columns: {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")

### 1.4 Cardinality Analysis

In [ ]:
print("\n1.4 Cardinality Analysis")

cardinality = pd.DataFrame({
    "column": application_train.columns,
    "unique_values": [application_train[c].nunique(dropna=False) for c in application_train.columns]
}).sort_values("unique_values", ascending=False)

print("\nTop 20 columns by unique values:")
display(cardinality.head(20))

print("\nCategorical columns and their unique values:")
for col in categorical_cols[:10]:
    print(f"\n{col}:")
    print(application_train[col].value_counts(dropna=False).head(10))

### 1.5 Duplicate Records Check

In [ ]:
print("\n1.5 Duplicate Records Check")

print(f"Duplicate rows in application_train: {application_train.duplicated().sum()}")
print(f"Duplicate SK_ID_CURR in application_train: {application_train['SK_ID_CURR'].duplicated().sum()}")

for name in ["bureau", "previous_application", "installments_payments", "POS_CASH_balance", "credit_card_balance"]:
    df = datasets[name]
    print(f"Duplicate SK_ID_CURR in {name}: {df['SK_ID_CURR'].duplicated().sum():,} (expected)")

### 1.6 Table Relationships Verification

In [ ]:
print("\n1.6 Table Relationships Verification")

train_ids = set(application_train["SK_ID_CURR"])
print(f"Training customers: {len(train_ids):,}")

coverage = []
for name, df in [
    ("bureau", bureau),
    ("previous_application", previous_application),
    ("installments_payments", installments),
    ("POS_CASH_balance", pos_cash),
    ("credit_card_balance", credit_card)
]:
    ids = set(df["SK_ID_CURR"])
    matched = len(train_ids.intersection(ids))
    coverage.append({
        "Dataset": name,
        "Customers Matched": matched,
        "Training Customers": len(train_ids),
        "Coverage %": round(matched / len(train_ids) * 100, 2)
    })

coverage_df = pd.DataFrame(coverage)
print("\nTable 2: Data Coverage - percentage of customers with historical data")
display(coverage_df)

## Part 2: Data Quality Analysis

### 2.1 Missing Value Analysis

In [ ]:
print("\n" + "="*80)
print("PART 2: DATA QUALITY ANALYSIS")
print("="*80)

print("\n2.1 Missing Value Analysis - Application Train")

missing = pd.DataFrame({
    "Missing Count": application_train.isnull().sum(),
    "Missing %": application_train.isnull().mean() * 100
})
missing = missing.sort_values("Missing %", ascending=False)

print("\nTop 50 variables by missingness:")
display(missing.head(50))

In [ ]:
missing_plot = missing[missing["Missing %"] > 5].head(30)
if len(missing_plot) > 0:
    plt.figure(figsize=(12, 10))
    sns.barplot(x=missing_plot["Missing %"], y=missing_plot.index)
    plt.title("Variables with Highest Missingness (>5%)")
    plt.xlabel("Missing values (%)")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

### 2.2 Missingness as Risk Signal

In [ ]:
print("\n2.2 Missingness as Risk Signal - EXT_SOURCE_1")

application_train["EXT_SOURCE_1_MISSING"] = application_train["EXT_SOURCE_1"].isna().astype(int)

missing_risk = application_train.groupby("EXT_SOURCE_1_MISSING")["TARGET"].agg(["count", "mean"])
missing_risk["Default Rate %"] = missing_risk["mean"] * 100
print("\nDefault rate by EXT_SOURCE_1 missing status:")
display(missing_risk)

for col in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]:
    app_train_copy = application_train.copy()
    app_train_copy[f"{col}_MISSING"] = app_train_copy[col].isna().astype(int)
    result = app_train_copy.groupby(f"{col}_MISSING")["TARGET"].agg(["count", "mean"])
    result["Default Rate %"] = result["mean"] * 100
    print(f"\n{col} missing status vs default rate:")
    display(result)

### 2.3 Numeric Summary with Percentiles

In [ ]:
print("\n2.3 Numeric Summary with Percentiles")

key_numeric = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
               "DAYS_BIRTH", "DAYS_EMPLOYED", "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]

summary = application_train[key_numeric].describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99])
print("\nSummary statistics for key numeric variables:")
display(summary)

### 2.4 Suspicious or Anomalous Values

In [ ]:
print("\n2.4 Anomalous Values Detection")

day_columns = [c for c in application_train.columns if "DAYS" in c]
print(f"\nFound {len(day_columns)} DAYS columns")
print("\nDAYS columns summary:")
display(application_train[day_columns].describe().T)

print("\nDAYS_EMPLOYED value distribution (top 10):")
display(application_train["DAYS_EMPLOYED"].value_counts().head(10))

print("\nExtreme values in key variables:")
for col in ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY"]:
    q99 = application_train[col].quantile(0.99)
    count_extreme = (application_train[col] > q99).sum()
    print(f"{col}: {count_extreme:,} values > 99th percentile ({q99:.0f})")

## Part 3: Target Analysis

### 3.1 Target Distribution

In [ ]:
print("\n" + "="*80)
print("PART 3: TARGET ANALYSIS")
print("="*80)

print("\n3.1 Target Distribution")

target_counts = application_train["TARGET"].value_counts()
target_percent = application_train["TARGET"].value_counts(normalize=True) * 100

print("Target Distribution:")
print(f"Class 0 (No Payment Difficulty): {target_counts[0]:,} ({target_percent[0]:.2f}%)")
print(f"Class 1 (Payment Difficulty): {target_counts[1]:,} ({target_percent[1]:.2f}%)")

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(data=application_train, x="TARGET")
plt.xticks([0, 1], ["No Payment Difficulty", "Payment Difficulty"])
plt.title("Distribution of Target Variable (Payment Difficulty)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

print(f"\nClass Imbalance Ratio: {target_percent[0]/target_percent[1]:.2f}:1")
print("This justifies using ROC-AUC, Precision, Recall, and F1 rather than accuracy")

### 3.2 Train vs Test Comparison

In [ ]:
print("\n3.2 Train vs Test Comparison")

print(f"Train shape: {application_train.shape}")
print(f"Test shape: {application_test.shape}")

train_features = set(application_train.columns) - {"TARGET"}
test_features = set(application_test.columns)

print(f"\nFeatures only in train: {train_features - test_features}")
print(f"Features only in test: {test_features - train_features}")

In [ ]:
train_missing = application_train.drop(columns="TARGET").isna().mean() * 100
test_missing = application_test.isna().mean() * 100

missing_compare = pd.DataFrame({
    "Train Missing %": train_missing,
    "Test Missing %": test_missing
})
missing_compare["Difference"] = abs(missing_compare["Train Missing %"] - missing_compare["Test Missing %"])
missing_compare = missing_compare.sort_values("Difference", ascending=False)

print("\nTop 20 variables with largest train/test missingness difference:")
display(missing_compare.head(20))

### 3.3 Train vs Test Distribution Comparison

In [ ]:
print("\n3.3 Train vs Test Distribution Comparison")

compare_cols = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "DAYS_BIRTH",
                "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]

for col in compare_cols:
    plt.figure(figsize=(8, 4))
    sns.kdeplot(application_train[col].dropna(), label="Train", alpha=0.6)
    sns.kdeplot(application_test[col].dropna(), label="Test", alpha=0.6)
    plt.title(f"Train vs Test Distribution: {col}")
    plt.legend()
    plt.tight_layout()
    plt.show()

## Part 4: Demographic EDA

### 4.1 Age Analysis

In [ ]:
print("\n" + "="*80)
print("PART 4: DEMOGRAPHIC EDA")
print("="*80)

print("\n4.1 Age Analysis")

application_train["AGE_YEARS"] = -application_train["DAYS_BIRTH"] / 365.25

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=application_train, x="AGE_YEARS", hue="TARGET", bins=40,
             stat="density", common_norm=False, alpha=0.6)
plt.title("Age Distribution by Repayment Outcome")
plt.xlabel("Age (years)")
plt.ylabel("Density")
plt.tight_layout()
plt.show()

In [ ]:
application_train["AGE_GROUP"] = pd.cut(application_train["AGE_YEARS"],
                                         bins=[20, 30, 40, 50, 60, 70, 100])

age_risk = application_train.groupby("AGE_GROUP", observed=False).agg(
    Customers=("TARGET", "size"),
    Default_Rate=("TARGET", "mean")
)
age_risk["Default_Rate"] = age_risk["Default_Rate"] * 100

print("\nDefault rate by age group:")
display(age_risk)

In [ ]:
plt.figure(figsize=(10, 6))
ax = age_risk["Default_Rate"].plot(kind="bar")
plt.title("Payment Difficulty Rate by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Default Rate (%)")
plt.xticks(rotation=45)
for i, v in enumerate(age_risk["Default_Rate"]):
    ax.text(i, v + 0.2, f"{v:.1f}%", ha='center')
plt.tight_layout()
plt.show()

### 4.2 Gender Analysis

In [ ]:
print("\n4.2 Gender Analysis")

gender_risk = application_train.groupby("CODE_GENDER")["TARGET"].agg(["count", "mean"])
gender_risk["Default Rate %"] = gender_risk["mean"] * 100
print("\nDefault rate by gender:")
display(gender_risk)

### 4.3 Categorical Analysis

In [ ]:
print("\n4.3 Categorical Analysis")

def category_target_summary(df, column):
    result = df.groupby(column, dropna=False)["TARGET"].agg(
        Customers="size",
        Defaults="sum",
        Default_Rate="mean"
    ).sort_values("Default_Rate", ascending=False)
    result["Default_Rate"] = result["Default_Rate"] * 100
    return result

def plot_default_rate(df, column, min_count=100):
    temp = df.groupby(column)["TARGET"].agg(
        Count="size",
        Default_Rate="mean"
    )
    temp = temp[temp["Count"] >= min_count]
    temp = temp.sort_values("Default_Rate", ascending=False)

    plt.figure(figsize=(10, 6))
    sns.barplot(data=temp.reset_index(), x="Default_Rate", y=column)
    plt.title(f"Payment Difficulty Rate by {column}")
    plt.xlabel("Default Rate (%)")
    plt.tight_layout()
    plt.show()
    return temp

In [ ]:
categorical_research_cols = [
    "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE", "OCCUPATION_TYPE", "ORGANIZATION_TYPE"
]

for col in categorical_research_cols:
    if col in application_train.columns:
        print(f"\n{col}:")
        display(category_target_summary(application_train, col).head(10))
        plot_default_rate(application_train, col)

## Part 5: Financial Affordability EDA

### 5.1 Create Financial Ratios

In [ ]:
print("\n" + "="*80)
print("PART 5: FINANCIAL AFFORDABILITY EDA")
print("="*80)

print("\n5.1 Creating Financial Ratios")

application_train["CREDIT_INCOME_RATIO"] = application_train["AMT_CREDIT"] / application_train["AMT_INCOME_TOTAL"]
application_train["ANNUITY_INCOME_RATIO"] = application_train["AMT_ANNUITY"] / application_train["AMT_INCOME_TOTAL"]
application_train["CREDIT_ANNUITY_RATIO"] = application_train["AMT_CREDIT"] / application_train["AMT_ANNUITY"]
application_train["INCOME_PER_PERSON"] = application_train["AMT_INCOME_TOTAL"] / application_train["CNT_FAM_MEMBERS"]
application_train["CREDIT_PER_PERSON"] = application_train["AMT_CREDIT"] / application_train["CNT_FAM_MEMBERS"]

### 5.2 Financial Features by Target

In [ ]:
print("\n5.2 Financial Features by Target")

financial_features = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY",
    "CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "CREDIT_ANNUITY_RATIO",
    "INCOME_PER_PERSON", "CREDIT_PER_PERSON"
]

financial_summary = application_train.groupby("TARGET")[financial_features].median().T
print("\nMedian values by target:")
display(financial_summary)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, col in enumerate(["CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "INCOME_PER_PERSON", "CREDIT_PER_PERSON"]):
    sns.boxplot(data=application_train, x="TARGET", y=col, ax=axes[idx])
    axes[idx].set_title(f"{col} by Target")
    axes[idx].set_xticklabels(["No Default", "Default"])

plt.tight_layout()
plt.show()

### 5.3 Annuity Burden Groups

In [ ]:
print("\n5.3 Annuity Burden Analysis")

application_train["ANNUITY_BURDEN_GROUP"] = pd.qcut(
    application_train["ANNUITY_INCOME_RATIO"],
    q=5,
    duplicates="drop"
)

burden_risk = application_train.groupby("ANNUITY_BURDEN_GROUP", observed=False).agg(
    Count=("TARGET", "size"),
    Default_Rate=("TARGET", "mean")
)
burden_risk["Default_Rate"] = burden_risk["Default_Rate"] * 100

print("\nDefault rate by annuity burden quintile:")
display(burden_risk)

In [ ]:
plt.figure(figsize=(10, 6))
burden_risk["Default_Rate"].plot(kind="bar")
plt.title("Payment Difficulty Rate by Annuity-to-Income Ratio Quintile")
plt.xlabel("Annuity Burden Quintile (5 = Highest Burden)")
plt.ylabel("Default Rate (%)")
plt.xticks(rotation=45)
for i, v in enumerate(burden_risk["Default_Rate"]):
    plt.text(i, v + 0.2, f"{v:.1f}%", ha='center')
plt.tight_layout()
plt.show()

### 5.4 Credit Burden Groups

In [ ]:
print("\n5.4 Credit Burden Analysis")

application_train["CREDIT_BURDEN_GROUP"] = pd.qcut(
    application_train["CREDIT_INCOME_RATIO"],
    q=5,
    duplicates="drop"
)

credit_burden_risk = application_train.groupby("CREDIT_BURDEN_GROUP", observed=False).agg(
    Count=("TARGET", "size"),
    Default_Rate=("TARGET", "mean")
)
credit_burden_risk["Default_Rate"] = credit_burden_risk["Default_Rate"] * 100

print("\nDefault rate by credit-to-income quintile:")
display(credit_burden_risk)

In [ ]:
plt.figure(figsize=(10, 6))
credit_burden_risk["Default_Rate"].plot(kind="bar")
plt.title("Payment Difficulty Rate by Credit-to-Income Ratio Quintile")
plt.xlabel("Credit Burden Quintile (5 = Highest Burden)")
plt.ylabel("Default Rate (%)")
plt.xticks(rotation=45)
for i, v in enumerate(credit_burden_risk["Default_Rate"]):
    plt.text(i, v + 0.2, f"{v:.1f}%", ha='center')
plt.tight_layout()
plt.show()

### 5.5 Goods Price vs Credit

In [ ]:
print("\n5.5 Goods Price vs Credit")

plt.figure(figsize=(8, 6))
sns.scatterplot(data=application_train.sample(10000), x="AMT_GOODS_PRICE", y="AMT_CREDIT",
                hue="TARGET", alpha=0.3)
plt.title("Credit Amount vs Goods Price")
plt.xlabel("Goods Price")
plt.ylabel("Credit Amount")
plt.tight_layout()
plt.show()

## Part 6: External Credit Scores

### 6.1 External Score Distributions

In [ ]:
print("\n" + "="*80)
print("PART 6: EXTERNAL CREDIT SCORES")
print("="*80)

print("\n6.1 External Score Distributions")

ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for idx, col in enumerate(ext_cols):
    sns.kdeplot(data=application_train, x=col, hue="TARGET", common_norm=False, ax=axes[idx])
    axes[idx].set_title(f"{col} Distribution by Target")
    axes[idx].set_xlabel("Score")
axes[0].legend(["No Default", "Default"])
plt.tight_layout()
plt.show()

### 6.2 Quantile Analysis

In [ ]:
print("\n6.2 External Score Quantile Analysis")

for col in ext_cols:
    application_train[f"{col}_Q"] = pd.qcut(application_train[col], q=5, duplicates="drop")
    result = application_train.groupby(f"{col}_Q", observed=False)["TARGET"].agg(
        Count="size", Default_Rate="mean"
    )
    result["Default_Rate"] = result["Default_Rate"] * 100
    print(f"\n{col} - Default rate by quintile:")
    display(result)

### 6.3 Combined External Score

In [ ]:
print("\n6.3 Combined External Score")

application_train["EXT_SOURCE_STD"] = application_train[ext_cols].std(axis=1)
application_train["EXT_SOURCE_AVAILABLE"] = application_train[ext_cols].notna().sum(axis=1)
application_train["EXT_SOURCE_MEAN"] = application_train[ext_cols].mean(axis=1)

### 6.4 Combined Score vs Default

In [ ]:
print("\n6.4 Combined Score Analysis")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.kdeplot(data=application_train, x="EXT_SOURCE_MEAN", hue="TARGET", common_norm=False, ax=axes[0])
axes[0].set_title("Mean External Score by Target")

sns.kdeplot(data=application_train, x="EXT_SOURCE_STD", hue="TARGET", common_norm=False, ax=axes[1])
axes[1].set_title("External Score Std Dev by Target")

sns.countplot(data=application_train, x="EXT_SOURCE_AVAILABLE", hue="TARGET", ax=axes[2])
axes[2].set_title("Number of Available Scores by Target")

plt.tight_layout()
plt.show()

### 6.5 Bureau Enquiries

In [ ]:
print("\n6.5 Bureau Enquiries")

bureau_enquiry_cols = [
    "AMT_REQ_CREDIT_BUREAU_HOUR",
    "AMT_REQ_CREDIT_BUREAU_DAY",
    "AMT_REQ_CREDIT_BUREAU_WEEK",
    "AMT_REQ_CREDIT_BUREAU_MON",
    "AMT_REQ_CREDIT_BUREAU_QRT",
    "AMT_REQ_CREDIT_BUREAU_YEAR"
]

application_train["TOTAL_BUREAU_ENQUIRIES"] = application_train[bureau_enquiry_cols].sum(axis=1)

enquiry_risk = application_train.groupby("TOTAL_BUREAU_ENQUIRIES")["TARGET"].agg(
    Count="size", Default_Rate="mean"
)
enquiry_risk["Default_Rate"] = enquiry_risk["Default_Rate"] * 100

print("\nDefault rate by number of bureau enquiries (top 10):")
display(enquiry_risk.head(10))

In [ ]:
plt.figure(figsize=(10, 6))
enquiry_risk[enquiry_risk["Count"] >= 100]["Default_Rate"].plot(kind="bar")
plt.title("Default Rate by Number of Bureau Enquiries (min 100 customers)")
plt.xlabel("Number of Enquiries")
plt.ylabel("Default Rate (%)")
plt.tight_layout()
plt.show()

## Part 7: Bureau EDA

### 7.1 Bureau Overview

In [ ]:
print("\n" + "="*80)
print("PART 7: BUREAU EDA")
print("="*80)

print("\n7.1 Bureau Data Overview")

print("Bureau dataset shape:", bureau.shape)
print("\nBureau column types:")
display(bureau.dtypes.value_counts())
print("\nBureau first few rows:")
display(bureau.head())

print("\nCREDIT_ACTIVE distribution:")
display(bureau["CREDIT_ACTIVE"].value_counts(dropna=False))

print("\nCREDIT_TYPE distribution (top 10):")
display(bureau["CREDIT_TYPE"].value_counts().head(10))

### 7.2 Bureau Loan Count

In [ ]:
print("\n7.2 Bureau Loan Count")

bureau_count = bureau.groupby("SK_ID_CURR").size().rename("BUREAU_LOAN_COUNT")
application_train = application_train.merge(bureau_count, on="SK_ID_CURR", how="left")

print("\nBureau loan count statistics:")
display(application_train.groupby("TARGET")["BUREAU_LOAN_COUNT"].describe())

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=application_train, x="TARGET", y="BUREAU_LOAN_COUNT")
plt.title("Bureau Loan Count by Target")
plt.xticks([0, 1], ["No Default", "Default"])
plt.tight_layout()
plt.show()

### 7.3 Bureau Financial Features

In [ ]:
print("\n7.3 Bureau Financial Features")

bureau_financial = bureau.groupby("SK_ID_CURR").agg(
    BUREAU_TOTAL_CREDIT=("AMT_CREDIT_SUM", "sum"),
    BUREAU_TOTAL_DEBT=("AMT_CREDIT_SUM_DEBT", "sum"),
    BUREAU_OVERDUE_AMT=("AMT_CREDIT_SUM_OVERDUE", "sum"),
    BUREAU_LOAN_COUNT=("SK_ID_BUREAU", "count"),
    BUREAU_ACTIVE_LOANS=("CREDIT_ACTIVE", lambda x: (x == "Active").sum()),
    BUREAU_MAX_DAYS_OVERDUE=("CREDIT_DAY_OVERDUE", "max"),
    BUREAU_PROLONG_COUNT=("CNT_CREDIT_PROLONG", "sum")
).reset_index()

bureau_financial["BUREAU_DEBT_CREDIT_RATIO"] = (
    bureau_financial["BUREAU_TOTAL_DEBT"] / bureau_financial["BUREAU_TOTAL_CREDIT"]
).replace([np.inf, -np.inf], np.nan)

### 7.4 Bureau Recency

In [ ]:
print("\n7.4 Bureau Recency")

bureau_recency = bureau.groupby("SK_ID_CURR").agg(
    MOST_RECENT_BUREAU_CREDIT=("DAYS_CREDIT", "max"),
    AVG_BUREAU_CREDIT_AGE=("DAYS_CREDIT", "mean")
).reset_index()

### 7.5 Merge Bureau Features

In [ ]:
print("\n7.5 Merging Bureau Features")

application_train = application_train.merge(bureau_financial, on="SK_ID_CURR", how="left")
application_train = application_train.merge(bureau_recency, on="SK_ID_CURR", how="left")

print(f"Updated application_train shape: {application_train.shape}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(data=application_train, x="BUREAU_DEBT_CREDIT_RATIO", hue="TARGET", common_norm=False)
plt.title("Bureau Debt-to-Credit Ratio by Target")
plt.xlim(0, 2)
plt.tight_layout()
plt.show()

In [ ]:
application_train["BUREAU_DEBT_GROUP"] = pd.qcut(
    application_train["BUREAU_DEBT_CREDIT_RATIO"],
    q=5,
    duplicates="drop"
)

bureau_debt_risk = application_train.groupby("BUREAU_DEBT_GROUP", observed=False)["TARGET"].agg(
    Count="size", Default_Rate="mean"
)
bureau_debt_risk["Default_Rate"] = bureau_debt_risk["Default_Rate"] * 100

print("\nDefault rate by bureau debt-to-credit quintile:")
display(bureau_debt_risk)

## Part 8: Bureau Balance EDA

### 8.1 Bureau Balance Status

In [ ]:
print("\n" + "="*80)
print("PART 8: BUREAU BALANCE EDA")
print("="*80)

print("\n8.1 Bureau Balance Status")

print("STATUS distribution:")
display(bureau_balance["STATUS"].value_counts(normalize=True, dropna=False))

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(data=bureau_balance, x="STATUS")
plt.title("Bureau Balance Status Distribution")
plt.xlabel("Status")
plt.tight_layout()
plt.show()

### 8.2 Delinquency Indicators

In [ ]:
print("\n8.2 Bureau Balance Delinquency Indicators")

delinquency_statuses = ["1", "2", "3", "4", "5"]
bureau_balance["HAS_DPD"] = bureau_balance["STATUS"].astype(str).isin(delinquency_statuses).astype(int)

severity_map = {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "5": 5, "C": np.nan, "X": np.nan}
bureau_balance["DPD_SEVERITY"] = bureau_balance["STATUS"].astype(str).map(severity_map)

### 8.3 Bureau Balance Features

In [ ]:
print("\n8.3 Bureau Balance Features")

bureau_balance_features = bureau_balance.groupby("SK_ID_BUREAU").agg(
    BUREAU_MONTHS_OBSERVED=("MONTHS_BALANCE", "count"),
    BUREAU_DPD_MONTHS=("HAS_DPD", "sum"),
    BUREAU_DPD_RATE=("HAS_DPD", "mean"),
    BUREAU_MAX_SEVERITY=("DPD_SEVERITY", "max")
).reset_index()

bureau_with_balance = bureau[["SK_ID_CURR", "SK_ID_BUREAU"]].merge(
    bureau_balance_features, on="SK_ID_BUREAU", how="left"
)

bureau_balance_customer = bureau_with_balance.groupby("SK_ID_CURR").agg(
    BUREAU_AVG_DPD_RATE=("BUREAU_DPD_RATE", "mean"),
    BUREAU_MAX_DPD_RATE=("BUREAU_DPD_RATE", "max"),
    BUREAU_HAS_DPD_HISTORY=("BUREAU_DPD_MONTHS", lambda x: (x > 0).sum()),
    BUREAU_MAX_SEVERITY=("BUREAU_MAX_SEVERITY", "max")
).reset_index()

application_train = application_train.merge(bureau_balance_customer, on="SK_ID_CURR", how="left")

print(f"Updated application_train shape: {application_train.shape}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(data=application_train, x="BUREAU_AVG_DPD_RATE", hue="TARGET", common_norm=False)
plt.title("Bureau Average DPD Rate by Target")
plt.tight_layout()
plt.show()

## Part 9: Previous Application EDA

### 9.1 Contract Status

In [ ]:
print("\n" + "="*80)
print("PART 9: PREVIOUS APPLICATION EDA")
print("="*80)

print("\n9.1 Contract Status")

print("Contract status distribution:")
display(previous_application["NAME_CONTRACT_STATUS"].value_counts(normalize=True))

previous_application["WAS_APPROVED"] = (previous_application["NAME_CONTRACT_STATUS"] == "Approved").astype(int)
previous_application["WAS_REFUSED"] = (previous_application["NAME_CONTRACT_STATUS"] == "Refused").astype(int)
previous_application["CREDIT_GRANTED_RATIO"] = (
    previous_application["AMT_CREDIT"] / previous_application["AMT_APPLICATION"]
)

### 9.2 Previous Application Features

In [ ]:
print("\n9.2 Previous Application Features")

previous_features = previous_application.groupby("SK_ID_CURR").agg(
    PREV_APPLICATION_COUNT=("SK_ID_PREV", "count"),
    PREV_APPROVAL_RATE=("WAS_APPROVED", "mean"),
    PREV_REFUSAL_RATE=("WAS_REFUSED", "mean"),
    AVG_PREV_APPLICATION_AMT=("AMT_APPLICATION", "mean"),
    AVG_PREV_CREDIT_AMT=("AMT_CREDIT", "mean"),
    AVG_CREDIT_GRANTED_RATIO=("CREDIT_GRANTED_RATIO", "mean"),
    MOST_RECENT_PREV_APPLICATION=("DAYS_DECISION", "max")
).reset_index()

### 9.3 Merge Previous Features

In [ ]:
print("\n9.3 Merging Previous Features")

application_train = application_train.merge(previous_features, on="SK_ID_CURR", how="left")
print(f"Updated application_train shape: {application_train.shape}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

sns.boxplot(data=application_train, x="TARGET", y="PREV_APPLICATION_COUNT", ax=axes[0,0])
axes[0,0].set_title("Previous Application Count by Target")
sns.boxplot(data=application_train, x="TARGET", y="PREV_REFUSAL_RATE", ax=axes[0,1])
axes[0,1].set_title("Previous Refusal Rate by Target")
sns.boxplot(data=application_train, x="TARGET", y="AVG_CREDIT_GRANTED_RATIO", ax=axes[1,0])
axes[1,0].set_title("Avg Credit Granted Ratio by Target")
sns.boxplot(data=application_train, x="TARGET", y="PREV_APPROVAL_RATE", ax=axes[1,1])
axes[1,1].set_title("Previous Approval Rate by Target")

plt.tight_layout()
plt.show()

## Part 10: Installment Payment EDA

### 10.1 Calculate Payment Delay

In [ ]:
print("\n" + "="*80)
print("PART 10: INSTALLMENT PAYMENT EDA")
print("="*80)

print("\n10.1 Calculating Payment Delay")

installments["DAYS_LATE"] = installments["DAYS_ENTRY_PAYMENT"] - installments["DAYS_INSTALMENT"]
installments["LATE_PAYMENT"] = (installments["DAYS_LATE"] > 0).astype(int)
installments["UNDERPAID"] = (installments["AMT_PAYMENT"] < installments["AMT_INSTALMENT"]).astype(int)
installments["PAYMENT_RATIO"] = installments["AMT_PAYMENT"] / installments["AMT_INSTALMENT"]
installments["PAYMENT_SHORTFALL"] = installments["AMT_INSTALMENT"] - installments["AMT_PAYMENT"]

### 10.2 Payment Delay Distribution

In [ ]:
print("\n10.2 Payment Delay Distribution")

print("DAYS_LATE statistics:")
display(installments["DAYS_LATE"].describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]))

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(data=installments, x="DAYS_LATE", bins=50)
plt.title("Payment Delay Distribution")
plt.xlabel("Days Late")
plt.xlim(-200, 200)

plt.subplot(1, 2, 2)
sns.boxplot(data=installments, y="DAYS_LATE")
plt.title("Payment Delay Boxplot")
plt.ylim(-200, 200)

plt.tight_layout()
plt.show()

### 10.3 Customer-Level Installment Features

In [ ]:
print("\n10.3 Customer-Level Installment Features")

installment_features = installments.groupby("SK_ID_CURR").agg(
    INSTALLMENT_COUNT=("NUM_INSTALMENT_NUMBER", "count"),
    PREVIOUS_LOANS_PAID=("SK_ID_PREV", "nunique"),
    AVG_DAYS_LATE=("DAYS_LATE", "mean"),
    MAX_DAYS_LATE=("DAYS_LATE", "max"),
    STD_DAYS_LATE=("DAYS_LATE", "std"),
    LATE_PAYMENT_RATE=("LATE_PAYMENT", "mean"),
    MAX_LATE_PAYMENT=("LATE_PAYMENT", "max"),
    UNDERPAYMENT_RATE=("UNDERPAID", "mean"),
    AVG_PAYMENT_RATIO=("PAYMENT_RATIO", "mean"),
    TOTAL_SHORTFALL=("PAYMENT_SHORTFALL", "sum")
).reset_index()

### 10.4 Recent Payment Behaviour

In [ ]:
print("\n10.4 Recent Payment Behaviour")

installments_sorted = installments.sort_values(
    ["SK_ID_CURR", "DAYS_INSTALMENT"],
    ascending=[True, False]
)

recent_3 = installments_sorted.groupby("SK_ID_CURR").head(3)

recent_features = recent_3.groupby("SK_ID_CURR").agg(
    RECENT3_LATE_RATE=("LATE_PAYMENT", "mean"),
    RECENT3_AVG_DAYS_LATE=("DAYS_LATE", "mean"),
    RECENT3_UNDERPAYMENT_RATE=("UNDERPAID", "mean")
).reset_index()

### 10.5 Repayment Deterioration Signal

In [ ]:
print("\n10.5 Repayment Deterioration Signal")

repayment_features = installment_features.merge(
    recent_features, on="SK_ID_CURR", how="left"
)

repayment_features["LATE_RATE_CHANGE"] = (
    repayment_features["RECENT3_LATE_RATE"] - repayment_features["LATE_PAYMENT_RATE"]
)

### 10.6 Merge Installment Features

In [ ]:
print("\n10.6 Merging Installment Features")

application_train = application_train.merge(repayment_features, on="SK_ID_CURR", how="left")
print(f"Updated application_train shape: {application_train.shape}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(data=application_train, x="LATE_PAYMENT_RATE", hue="TARGET", common_norm=False)
plt.title("Late Payment Rate by Target")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(data=application_train, x="LATE_RATE_CHANGE", hue="TARGET", common_norm=False)
plt.title("Late Rate Change (Recent - Historical) by Target")
plt.axvline(0, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Part 11: POS Cash EDA

### 11.1 POS Cash Features

In [ ]:
print("\n" + "="*80)
print("PART 11: POS CASH EDA")
print("="*80)

print("\n11.1 POS Cash Features")

pos_cash["HAS_POS_DPD"] = (pos_cash["SK_DPD"] > 0).astype(int)

pos_features = pos_cash.groupby("SK_ID_CURR").agg(
    POS_MONTHS=("MONTHS_BALANCE", "count"),
    AVG_POS_DPD=("SK_DPD", "mean"),
    MAX_POS_DPD=("SK_DPD", "max"),
    POS_DPD_RATE=("HAS_POS_DPD", "mean"),
    AVG_INSTALLMENTS_LEFT=("CNT_INSTALMENT_FUTURE", "mean")
).reset_index()

In [ ]:
application_train = application_train.merge(pos_features, on="SK_ID_CURR", how="left")
print(f"Updated application_train shape: {application_train.shape}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(data=application_train, x="POS_DPD_RATE", hue="TARGET", common_norm=False)
plt.title("POS DPD Rate by Target")
plt.tight_layout()
plt.show()

## Part 12: Credit Card EDA

### 12.1 Credit Card Features

In [ ]:
print("\n" + "="*80)
print("PART 12: CREDIT CARD EDA")
print("="*80)

print("\n12.1 Credit Card Features")

credit_card["UTILIZATION"] = (
    credit_card["AMT_BALANCE"] / credit_card["AMT_CREDIT_LIMIT_ACTUAL"]
).replace([np.inf, -np.inf], np.nan)

credit_card["PAYMENT_TO_MIN_RATIO"] = (
    credit_card["AMT_PAYMENT_CURRENT"] / credit_card["AMT_INST_MIN_REGULARITY"]
).replace([np.inf, -np.inf], np.nan)

credit_card["ATM_DRAWING_SHARE"] = (
    credit_card["AMT_DRAWINGS_ATM_CURRENT"] / credit_card["AMT_DRAWINGS_CURRENT"]
).replace([np.inf, -np.inf], np.nan)

credit_card["HAS_CC_DPD"] = (credit_card["SK_DPD"] > 0).astype(int)

### 12.2 Aggregate Credit Card Features

In [ ]:
print("\n12.2 Aggregate Credit Card Features")

cc_features = credit_card.groupby("SK_ID_CURR").agg(
    CC_MONTHS_OBSERVED=("MONTHS_BALANCE", "count"),
    AVG_CC_BALANCE=("AMT_BALANCE", "mean"),
    AVG_CC_UTILIZATION=("UTILIZATION", "mean"),
    MAX_CC_UTILIZATION=("UTILIZATION", "max"),
    AVG_CC_PAYMENT_MIN_RATIO=("PAYMENT_TO_MIN_RATIO", "mean"),
    MAX_CC_DPD=("SK_DPD", "max"),
    CC_DPD_RATE=("HAS_CC_DPD", "mean")
).reset_index()

In [ ]:
application_train = application_train.merge(cc_features, on="SK_ID_CURR", how="left")
print(f"Updated application_train shape: {application_train.shape}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(data=application_train, x="AVG_CC_UTILIZATION", hue="TARGET", common_norm=False)
plt.title("Average Credit Card Utilisation by Target")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

## Part 13: History Availability as Risk Signal

### 13.1 History Availability Indicators

In [ ]:
print("\n" + "="*80)
print("PART 13: HISTORY AVAILABILITY AS RISK SIGNAL")
print("="*80)

print("\n13.1 History Availability Indicators")

application_train["HAS_BUREAU_HISTORY"] = application_train["SK_ID_CURR"].isin(
    bureau["SK_ID_CURR"]
).astype(int)

application_train["HAS_PREVIOUS_APPLICATION"] = application_train["SK_ID_CURR"].isin(
    previous_application["SK_ID_CURR"]
).astype(int)

application_train["HAS_INSTALLMENT_HISTORY"] = application_train["SK_ID_CURR"].isin(
    installments["SK_ID_CURR"]
).astype(int)

application_train["HAS_POS_HISTORY"] = application_train["SK_ID_CURR"].isin(
    pos_cash["SK_ID_CURR"]
).astype(int)

application_train["HAS_CC_HISTORY"] = application_train["SK_ID_CURR"].isin(
    credit_card["SK_ID_CURR"]
).astype(int)

### 13.2 History Availability vs Default

In [ ]:
print("\n13.2 History Availability vs Default")

history_flags = [
    "HAS_BUREAU_HISTORY",
    "HAS_PREVIOUS_APPLICATION",
    "HAS_INSTALLMENT_HISTORY",
    "HAS_POS_HISTORY",
    "HAS_CC_HISTORY"
]

for col in history_flags:
    result = application_train.groupby(col)["TARGET"].agg(
        Count="size",
        Default_Rate="mean"
    )
    result["Default_Rate"] = result["Default_Rate"] * 100
    print(f"\n{col}:")
    display(result)

## Part 14: Interaction EDA

### 14.1 Late Payment Rate x Debt Burden

In [ ]:
print("\n" + "="*80)
print("PART 14: INTERACTION EDA")
print("="*80)

print("\n14.1 Late Payment Rate x Debt Burden")

application_train["LATE_RATE_GROUP"] = pd.qcut(
    application_train["LATE_PAYMENT_RATE"], q=4, duplicates="drop"
)

application_train["DEBT_GROUP"] = pd.qcut(
    application_train["CREDIT_INCOME_RATIO"], q=4, duplicates="drop"
)

interaction = application_train.groupby(
    ["LATE_RATE_GROUP", "DEBT_GROUP"], observed=False
)["TARGET"].mean().unstack()

plt.figure(figsize=(10, 7))
sns.heatmap(interaction, annot=True, fmt=".3f", cmap="Reds")
plt.title("Payment Difficulty Rate by Late Payment Rate and Debt Burden")
plt.xlabel("Debt Burden (Credit/Income)")
plt.ylabel("Late Payment Rate")
plt.tight_layout()
plt.show()

### 14.2 External Score x Late Payment Rate

In [ ]:
print("\n14.2 External Score x Late Payment Rate")

application_train["EXT_SCORE_GROUP"] = pd.qcut(
    application_train["EXT_SOURCE_MEAN"], q=4, duplicates="drop"
)

interaction_ext = application_train.groupby(
    ["EXT_SCORE_GROUP", "LATE_RATE_GROUP"], observed=False
)["TARGET"].mean().unstack()

plt.figure(figsize=(10, 7))
sns.heatmap(interaction_ext, annot=True, fmt=".3f", cmap="Reds")
plt.title("Payment Difficulty Rate by External Score and Late Payment Rate")
plt.xlabel("Late Payment Rate")
plt.ylabel("External Score")
plt.tight_layout()
plt.show()

## Part 15: Social Circle and Document Flags

### 15.1 Social Circle Risk

In [ ]:
print("\n" + "="*80)
print("PART 15: SOCIAL CIRCLE AND DOCUMENT FLAGS")
print("="*80)

print("\n15.1 Social Circle Risk")

application_train["SOCIAL_30_DEFAULT_RATE"] = (
    application_train["DEF_30_CNT_SOCIAL_CIRCLE"] /
    application_train["OBS_30_CNT_SOCIAL_CIRCLE"]
).replace([np.inf, -np.inf], np.nan)

application_train["SOCIAL_60_DEFAULT_RATE"] = (
    application_train["DEF_60_CNT_SOCIAL_CIRCLE"] /
    application_train["OBS_60_CNT_SOCIAL_CIRCLE"]
).replace([np.inf, -np.inf], np.nan)

### 15.2 Document Count

In [ ]:
print("\n15.2 Document Count")

document_cols = [c for c in application_train.columns if c.startswith("FLAG_DOCUMENT_")]
application_train["DOCUMENT_COUNT"] = application_train[document_cols].sum(axis=1)

doc_risk = application_train.groupby("DOCUMENT_COUNT")["TARGET"].agg(
    Count="size", Default_Rate="mean"
)
doc_risk["Default_Rate"] = doc_risk["Default_Rate"] * 100

print("\nDefault rate by document count:")
display(doc_risk)

## Part 16: Feature Ranking

### 16.1 Univariate AUC Ranking

In [ ]:
print("\n" + "="*80)
print("PART 16: FEATURE RANKING")
print("="*80)

print("\n16.1 Univariate AUC Ranking")

def calculate_univariate_auc(df, target="TARGET"):
    results = []
    numeric_cols = df.select_dtypes(include=np.number).columns

    for col in numeric_cols:
        if col in [target, "SK_ID_CURR"]:
            continue
        temp = df[[col, target]].dropna()
        if len(temp) < 100 or temp[col].nunique() < 2:
            continue
        try:
            auc = roc_auc_score(temp[target], temp[col])
            if auc >= 0.5:
                results.append({"Feature": col, "AUC": auc})
        except:
            continue

    return pd.DataFrame(results).sort_values("AUC", ascending=False)

auc_ranking = calculate_univariate_auc(application_train)
print("\nTop 30 features by Univariate AUC:")
display(auc_ranking.head(30))

### 16.2 Mutual Information Ranking

In [ ]:
print("\n16.2 Mutual Information Ranking")

numeric_research = application_train.select_dtypes(include=np.number).drop(
    columns=["TARGET", "SK_ID_CURR"], errors="ignore"
)

mi_df = numeric_research.copy()
mi_df = mi_df.replace([np.inf, -np.inf], np.nan)
mi_df = mi_df.fillna(mi_df.median())

mi_scores = mutual_info_classif(
    mi_df,
    application_train["TARGET"],
    random_state=42
)

mi_results = pd.DataFrame({
    "Feature": mi_df.columns,
    "Mutual Information": mi_scores
}).sort_values("Mutual Information", ascending=False)

print("\nTop 30 features by Mutual Information:")
display(mi_results.head(30))

### 16.3 Correlation with TARGET

In [ ]:
print("\n16.3 Correlation with TARGET")

correlations = application_train.select_dtypes(include=np.number).corr()["TARGET"].sort_values(ascending=False)
correlations = correlations[correlations.index != "TARGET"]

print("\nTop 30 features by Correlation with TARGET:")
display(correlations.head(30))

### 16.4 Missingness Risk Ranking

In [ ]:
print("\n16.4 Missingness Risk Ranking")

missing_risk_results = []
for col in application_train.columns:
    if application_train[col].isna().sum() > 0:
        missing_flag = application_train[col].isna().astype(int)
        try:
            auc = roc_auc_score(application_train["TARGET"], missing_flag)
            if auc >= 0.5:
                missing_risk_results.append({
                    "Feature": col,
                    "Missing %": application_train[col].isna().mean() * 100,
                    "AUC": auc
                })
        except:
            continue

missing_risk_df = pd.DataFrame(missing_risk_results).sort_values("AUC", ascending=False)

print("\nMissingness risk ranking (AUC > 0.5):")
display(missing_risk_df.head(20))

## Part 17: Summary and Conclusions

### 17.1 Final Dataset Shape

In [ ]:
print("\n" + "="*80)
print("PART 17: SUMMARY AND CONCLUSIONS")
print("="*80)

print("\n17.1 Final Dataset Summary")

print(f"Final application_train shape: {application_train.shape}")
print(f"Rows: {application_train.shape[0]:,}")
print(f"Columns: {application_train.shape[1]}")

print("\nTop 10 features by Univariate AUC:")
display(auc_ranking.head(10))

### 17.2 Key Findings

In [ ]:
print("\n17.2 Key Findings Summary:")
print("="*60)

print("\n1. Dataset Understanding:")
print(f"   - Total training samples: {application_train.shape[0]:,}")
print(f"   - Features: {application_train.shape[1]}")
print(f"   - Target imbalance: {target_percent[0]:.1f}% no default, {target_percent[1]:.1f}% default")

print("\n2. Data Coverage:")
for _, row in coverage_df.iterrows():
    print(f"   - {row['Dataset']}: {row['Coverage %']:.1f}% of customers")

print("\n3. Key Risk Factors Identified:")
top_features = auc_ranking.head(5)["Feature"].tolist()
for feat in top_features:
    print(f"   - {feat}")

print("\n4. Important Behavioral Features:")
behavioral_features = [f for f in auc_ranking.head(20)["Feature"].tolist()
                       if any(x in f.upper() for x in ["LATE", "DPD", "PAY", "INSTALL", "DELINQU"])]
for feat in behavioral_features[:5]:
    print(f"   - {feat}")

### 17.3 Save Final Dataset

In [ ]:
print("\n17.3 Saving final dataset...")

application_train.to_parquet(BASE_PATH / "research_dataset.parquet")
print("Dataset saved to:", BASE_PATH / "research_dataset.parquet")

print("\nEDA Complete!")
print("="*60)